check the engine

spark is a ready-made object called the SparkSession: your connection to the cluster. Databricks creates it for you. It should print 4.1.0.

In [0]:
print(spark.version)

read a table using the three-level name

catalog.schema.table is the Unity Catalog naming you saw earlier. Nothing visible happens: Spark is lazy. It only plans the work and doesn’t read data yet (an exam concept).

In [0]:
df = spark.table("samples.nyctaxi.trips")

look at it

display() is the Databricks version of pandas df.head(), but with a nice interactive table

In [0]:
display(df)

action → now Spark actually works

count() is an action, so Spark now reads the data. It’s the same as len(df) in pandas.
printSchema() shows columns and data types, like df.dtypes.

In [0]:
print(df.count())
df.printSchema()

a real analysis

Line by line:

functions as F is Spark’s toolbox of functions (count, avg, round…)
groupBy + agg works like SQL GROUP BY with COUNT(*) and AVG()
alias names the output column, like SQL AS
orderBy(F.desc(...)) works like ORDER BY trips DESC
limit(10) returns the top 10

In [0]:
from pyspark.sql import functions as F

result = (df.groupBy("pickup_zip")
            .agg(F.count("*").alias("trips"),
                 F.round(F.avg("fare_amount"), 2).alias("avg_fare"))
            .orderBy(F.desc("trips")))
display(result.limit(10))

the same thing in SQL (your strength)

%sql at the top switches that cell to SQL. Both cells give the same result, because PySpark and Spark SQL run on the same engine.

In [0]:
%sql
SELECT pickup_zip, COUNT(*) AS trips, ROUND(AVG(fare_amount), 2) AS avg_fare
FROM samples.nyctaxi.trips
GROUP BY pickup_zip
ORDER BY trips DESC
LIMIT 10

In [0]:
%sql
SELECT usage_date,
       sku_name,
       ROUND(SUM(usage_quantity), 2) AS dbus
FROM system.billing.usage
GROUP BY usage_date, sku_name
ORDER BY usage_date DESC